# sudregex tutorial notebook

This notebook walks through the main `sudregex` workflows:

- install and import the package
- load note data into a pandas DataFrame
- run `extract_df()` in memory
- inspect previews in a notebook
- compare serial and parallel backends
- run the file-based API
- handle headerless text files and custom separators

## 1. Installation

Install from PyPI:

```bash
pip install sud-regex
```

Or install from source in editable mode:

```bash
git clone https://github.com/quantitativenurse/sud-regex.git
cd sud-regex
python -m venv .venv
source .venv/bin/activate
pip install -U pip
pip install -e .[dev]
```

In [1]:
import pandas as pd
import sudregex as sud

print("sudregex version:", sud.__version__)

sudregex version: 0.1.6


## 2. Load a simple in-memory example

For `extract_df()`, your DataFrame should contain at minimum:

- a note identifier column
- a note text column

A person identifier column is optional.

In [10]:
df = pd.DataFrame(
    {
        "patient_id": ["P001", "P002", "P003", "P004"],
        "note_id": ["1001", "1002", "1003", "1004"],
        "note_text": [
            "Patient reports daily heroin use for the past 2 years.",
            "Denies opioid use. No heroin, fentanyl, or oxycodone use reported.",
            "Started buprenorphine-naloxone for opioid use disorder.",
            "Discharge instructions: take oxycodone only as prescribed for pain.",
        ],
    }
)

df

,patient_id,note_id,note_text
0,P001,1001,Patient reports daily heroin use for the past ...
1,P002,1002,"Denies opioid use. No heroin, fentanyl, or oxy..."
2,P003,1003,Started buprenorphine-naloxone for opioid use ...
3,P004,1004,Discharge instructions: take oxycodone only as...


## 3. Use packaged defaults

`sudregex` includes a default checklist and default term lists.

In [11]:
checklist = sud.checklist_abc
termslist = sud.default_termslist

type(checklist), type(termslist)

(dict, dict)

## 4. Run `extract_df()` with previews

In [12]:
result_df, previews_df = sud.extract_df(
    df=df,
    checklist=checklist,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    include_note_text=True,
    exclude_discharge_mentions=True,
    preview_count=5,
    preview_span=120,
    negation_scope="left",
    debug=False,
    return_previews_df=True,
)

print("result_df shape:", result_df.shape)
print("previews_df shape:", previews_df.shape)

result_df shape: (4, 72)
previews_df shape: (0, 6)


In [13]:
result_df.head()

,patient_id,note_id,illicit_drugs,illicit_drugs_SUBSTANCE_MATCHED,illicit_drugs_SUBSTANCE_MATCHED_NEG,problem_drinking,problem_drinking_NEG,dui,dui_NEG,hoarding,...,minimal_relief_x,minimal_relief_x_SUBSTANCE_MATCHED,tolerance,tolerance_SUBSTANCE_MATCHED,tolerance_SUBSTANCE_MATCHED_NEG,med_agreement,SO_concern,SO_concern_SUBSTANCE_MATCHED,SO_concern_SUBSTANCE_MATCHED_NEG,note_text
0,P001,1001,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Patient reports daily heroin use for the past ...
1,P002,1002,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,"Denies opioid use. No heroin, fentanyl, or oxy..."
2,P003,1003,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Started buprenorphine-naloxone for opioid use ...
3,P004,1004,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Discharge instructions: take oxycodone only as...


In [14]:
previews_df.head()

,item_key,note_id,span_start,span_end,snippet,snippet_marked


## 5. Filter previews for a single checklist item

In [15]:
if not previews_df.empty and "item_key" in previews_df.columns:
    display(previews_df[["item_key", "note_id", "snippet_marked"]].head(10))
else:
    print("No previews were generated for this sample.")

No previews were generated for this sample.


## 6. Try a different negation scope

Supported values:

- `left`
- `right`
- `both`

In [16]:
result_both = sud.extract_df(
    df=df,
    checklist=checklist,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    include_note_text=False,
    exclude_discharge_mentions=True,
    negation_scope="both",
    parallel=False,
)

result_both.head()

,patient_id,note_id,illicit_drugs,illicit_drugs_SUBSTANCE_MATCHED,illicit_drugs_SUBSTANCE_MATCHED_NEG,problem_drinking,problem_drinking_NEG,dui,dui_NEG,hoarding,...,lack_interest_rehab_SUBSTANCE_MATCHED,minimal_relief_x,minimal_relief_x_SUBSTANCE_MATCHED,tolerance,tolerance_SUBSTANCE_MATCHED,tolerance_SUBSTANCE_MATCHED_NEG,med_agreement,SO_concern,SO_concern_SUBSTANCE_MATCHED,SO_concern_SUBSTANCE_MATCHED_NEG
0,P001,1001,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,P002,1002,1,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,P003,1003,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,P004,1004,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 7. Use checklist and term-list files

You can pass filesystem paths instead of in-memory objects.

In [17]:
CHECKLIST_PATH = "path/to/checklist.py"
TERMSLIST_PATH = "path/to/termslist.py"

example = '''
result_df = sud.extract_df(
    df=df,
    checklist=CHECKLIST_PATH,
    termslist=TERMSLIST_PATH,
    terms_active="opioid_terms,alcohol_terms",
    person_column="patient_id",
    id_column="note_id",
)
'''
print(example)


result_df = sud.extract_df(
    df=df,
    checklist=CHECKLIST_PATH,
    termslist=TERMSLIST_PATH,
    terms_active="opioid_terms,alcohol_terms",
    person_column="patient_id",
    id_column="note_id",
)



## 8. Compare serial and parallel backends

`sudregex` supports:

- serial execution
- `pandarallel`
- `loky`

In [18]:
res_serial = sud.extract_df(
    df=df,
    checklist=checklist,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    parallel=False,
)

res_pandarallel = sud.extract_df(
    df=df,
    checklist=checklist,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    parallel=True,
    parallel_backend="pandarallel",
    n_workers=2,
)

res_loky = sud.extract_df(
    df=df,
    checklist=checklist,
    termslist=termslist,
    terms_active="opioid_terms",
    person_column="patient_id",
    id_column="note_id",
    parallel=True,
    parallel_backend="loky",
    n_workers=2,
)

print("Serial vs Pandarallel exact:", res_serial.equals(res_pandarallel))
print("Serial vs Loky exact:", res_serial.equals(res_loky))
print("Pandarallel vs Loky exact:", res_pandarallel.equals(res_loky))

INFO: Pandarallel will run on 2 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Serial vs Pandarallel exact: True
Serial vs Loky exact: True
Pandarallel vs Loky exact: True


## 9. Load a headerless text file with a custom separator

When using `pandas.read_csv(..., engine="python")`, the separator is treated
as a regular expression, so regex-special characters must be escaped correctly.

In [19]:
INPUT_PATH = "path/to/notes.txt"

df_headerless = pd.read_csv(
    INPUT_PATH,
    sep=r"\t!\^!\t",
    engine="python",
    header=None,
    names=["patient_id", "note_id", "note_text"],
    dtype="string",
)

df_headerless.head()

FileNotFoundError: [Errno 2] No such file or directory: 'path/to/notes.txt'

## 10. Run the file-based API from Python

In [20]:
sud.extract(
    in_file="path/to/notes.csv",
    out_file="path/to/results.csv",
    checklist="path/to/checklist.py",
    separator=",",
    termslist="path/to/termslist.py",
    terms_active="opioid_terms",
    include_note_text=False,
    exclude_discharge_mentions=True,
    preview_count=10,
    preview_file="note_previews.txt",
    preview_csv="previews.csv",
    negation_scope="left",
    parallel=True,
    parallel_backend="loky",
    n_workers=4,
)

FileNotFoundError: [Errno 2] No such file or directory: '/panfs/accrepfs.vampire/data/g_jeffery_lab/substance_use_disorders/code/preprocessing/regex/generalized/regex_package/path/to/checklist.py'

# Checklist Validation

In [32]:
import re
from sudregex.validation import validate_checklist


checklist = sud.checklist_abc
termslist = sud.default_termslist

if isinstance(termslist, dict):
    opioid_terms = termslist.get("opioid_terms", [])
else:
    opioid_terms = getattr(termslist, "opioid_terms", [])
    
    
examples_df = pd.DataFrame(
    [
        {
            "note_id": "OP001",
            "item_key": "4",
            "item_code": "4",
            "expected": 1,
            "note_text": (
                "Patient reports running out of oxycodone early again this month "
                "and asks for refill early because he has been taking extra doses."
            ),
        },
        {
            "note_id": "OP002",
            "item_key": "4",
            "item_code": "4",
            "expected": 1,
            "note_text": (
                "Hydrocodone prescription reviewed. He states he ran out early "
                "after using more tablets than prescribed."
            ),
        },
        {
            "note_id": "OP003",
            "item_key": "4",
            "item_code": "4",
            "expected": 0,
            "note_text": (
                "Patient denies running out early and says morphine has lasted "
                "exactly as prescribed."
            ),
        },
        {
            "note_id": "OP004",
            "item_key": "4",
            "item_code": "4",
            "expected": 0,
            "note_text": (
                "Discharge instructions reviewed. Do not refill early and do not "
                "exceed prescribed oxycodone dosing."
            ),
        },
        {
            "note_id": "OP005",
            "item_key": "4",
            "item_code": "4",
            "expected": 0,
            "note_text": (
                "Patient ran out early of blood pressure medication but is still "
                "taking tramadol as prescribed."
            ),
        },
        {
            "note_id": "OP006",
            "item_key": "4",
            "item_code": "4",
            "expected": 1,
            "note_text": (
                "She is out early on Percocet and requests to refill early before "
                "the scheduled date."
            ),
        },
    ]
)

examples_df


detailed, by_item= validate_checklist(
    checklist=checklist,
    examples=examples_df,
    substance_terms=opioid_terms,
)

print("Detailed shape:", detailed.shape)
print("By-item shape:", by_item.shape)


Detailed shape: (6, 13)
By-item shape: (2, 8)


In [33]:
detailed

,item_code,item_key,note_id,expected,note_text,actual_match,mismatch,substance_nearby,accepted_substance_nearby,common_fp_nearby,accepted_common_fp_nearby,raw_hits,failure_reason
0,4,4,OP001,1,Patient reports running out of oxycodone early...,1,0,1,1,0,0,2,
1,4,4,OP002,1,Hydrocodone prescription reviewed. He states h...,1,0,1,1,0,0,1,
2,4,4,OP003,0,Patient denies running out early and says morp...,0,0,1,0,0,0,1,negated
3,4,4,OP004,0,Discharge instructions reviewed. Do not refill...,0,0,1,0,0,0,1,negated
4,4,4,OP005,0,Patient ran out early of blood pressure medica...,1,1,1,1,0,0,1,
5,4,4,OP006,1,She is out early on Percocet and requests to r...,1,0,1,1,0,0,2,


In [25]:
detailed

,item_code,item_key,note_id,expected,note_text,actual_match,mismatch,raw_hits,failure_reason
0,foo_chk,foo_chk,N1,1,patient has foo today,1,0,1,
1,foo_chk,foo_chk,N2,0,patient does not foo today,0,0,1,negated
2,foo_chk,foo_chk,N3,1,foo noted again,1,0,1,


In [26]:
previews

,item_key,note_id,match_span_start,match_span_end,match_text,snippet,snippet_start,snippet_end,substance_term,negation_cue,common_fp_hit
0,foo_chk,N1,12,15,foo,patient has foo,0,15,None,None,None
1,foo_chk,N2,17,20,foo,patient does not foo,0,20,None,not,None
2,foo_chk,N3,0,3,foo,foo,0,3,None,None,None


## 11. CLI examples

### macOS / Linux

```bash
sudregex --extract \
  --in_file path/to/notes.csv \
  --out_file path/to/results.csv \
  --checklist path/to/checklist.py \
  --termslist path/to/termslist.py \
  --terms_active opioid_terms \
  --separator , \
  --parallel \
  --parallel-backend loky \
  --n-workers 4 \
  --person-column patient_id \
  --note-id-column note_id
```

### Windows PowerShell

```powershell
sudregex --extract `
  --in_file path\to\notes.csv `
  --out_file path\to\results.csv `
  --checklist path\to\checklist.py `
  --termslist path\to\termslist.py `
  --terms_active opioid_terms `
  --separator "," `
  --parallel `
  --parallel-backend loky `
  --n-workers 4 `
  --person-column patient_id `
  --note-id-column note_id
```

## 12. Headerless CLI examples with a custom separator

### macOS / Linux

```bash
sudregex --extract \
  --in_file path/to/notes.txt \
  --out_file path/to/results.csv \
  --separator $'\t!\\^!\t' \
  --checklist path/to/checklist.py \
  --termslist path/to/termslist.py \
  --terms_active opioid_terms \
  --parallel \
  --parallel-backend pandarallel \
  --n-workers 4 \
  --no-header \
  --columns patient_id,note_id,note_text
```

### Windows PowerShell

```powershell
sudregex --extract `
  --in_file path\to\notes.txt `
  --out_file path\to\results.csv `
  --separator "\t!\^!\t" `
  --checklist path\to\checklist.py `
  --termslist path\to\termslist.py `
  --terms_active opioid_terms `
  --parallel `
  --parallel-backend loky `
  --n-workers 4 `
  --no-header `
  --columns patient_id,note_id,note_text
```

## 13. Troubleshooting

- Escape regex-special characters in custom separators.
- If Pandarallel initializes but fails during execution, verify that the input parsed into non-empty note-text rows.
- If Loky is requested, ensure `joblib` is installed.
- For notebook QA, use `return_previews_df=True`.
- To validate backend consistency, compare outputs with `.equals(...)` or compare written CSV files.

## 14. Suggested validation workflow

1. run a small in-memory example
2. inspect `previews_df`
3. compare serial vs parallel outputs
4. test on a larger realistic subset
5. move to full-batch CLI execution